In [ ]:
from pathlib import Path

# -------------------------------------------------
# PROJECT ROOT (graphCeylon-API)
# -------------------------------------------------
REPO_ROOT = Path(__file__).resolve().parents[3]
# ↑ adjust if file depth differs (print it once to verify)

print("Repo root:", REPO_ROOT)

# -------------------------------------------------
# DATA PATHS
# -------------------------------------------------
DATA_DIR = REPO_ROOT / "data" / "recommender_service" / "clean"
INPUT_CSV = DATA_DIR / "sltda_arrivals_monthly.csv"

# -------------------------------------------------
# OUTPUT PATHS
# -------------------------------------------------
FORECAST_DIR = REPO_ROOT / "data" / "recommender_service" / "forecast"
MODELS_DIR = REPO_ROOT / "models"

FORECAST_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


New CWD: E:\Research_Final\component3_service
Loaded 94 monthly points from 2018-01-01 to 2025-10-01
[ARIMA] Validation window: 2024-11-01 → 2025-10-01
[ARIMA] Validation RMSE: 129858.25
[ARIMA] Validation MAE : 106764.25
Saved ARIMA model → models\arima_arrivals_1_1_1.pkl


In [ ]:
# -------------------------------------------
# ARIMA Arrivals Forecasting - Clean Version
# -------------------------------------------

import os


import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthBegin
from statsmodels.tsa.arima.model import ARIMA


# Paths
INPUT_CSV = DATA_DIR / "sltda_arrivals_monthly.csv"



def load_arrivals() -> pd.DataFrame:
    df = pd.read_csv(INPUT_CSV, parse_dates=["date"])
    df = df.sort_values("date")
    df = df.set_index("date")

    # Ensure DatetimeIndex with monthly-start freq
    df.index = pd.to_datetime(df.index)
    df = df.asfreq("MS")

    df["total_arrivals"] = df["total_arrivals"].astype(float)

    # Handle COVID collapse
    df.loc[df["total_arrivals"] < 1000, "total_arrivals"] = np.nan
    df["total_arrivals"] = df["total_arrivals"].interpolate(method="linear")

    print(
        f"Loaded {len(df)} monthly points from "
        f"{df.index.min().date()} to {df.index.max().date()}"
    )
    return df


def evaluate_arima(df: pd.DataFrame, val_months: int = 12):
    series = df["total_arrivals"]

    if len(series) <= val_months + 5:
        print("[ARIMA] Not enough data to split.")
        return None, None, None

    train = series.iloc[:-val_months]
    val   = series.iloc[-val_months:]

    model = ARIMA(train, order=(1, 1, 1))
    fitted = model.fit()

    forecast_vals = fitted.forecast(steps=val_months)

    y_true = val.values
    y_pred = forecast_vals.values

    mse = np.mean((y_true - y_pred)**2)
    rmse = math.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    # Robust date conversion
    start_date = pd.to_datetime(val.index.min()).date()
    end_date   = pd.to_datetime(val.index.max()).date()

    print(f"[ARIMA] Validation window: {start_date} → {end_date}")
    print(f"[ARIMA] Validation RMSE: {rmse:.2f}")
    print(f"[ARIMA] Validation MAE : {mae:.2f}")

    return rmse, mae, fitted


# --------- RUN EVERYTHING ---------

df_arrivals = load_arrivals()

rmse, mae, baseline_arima_model = evaluate_arima(df_arrivals, val_months=12)

if baseline_arima_model is not None:
    MODELS_DIR = Path("models")
    MODELS_DIR.mkdir(exist_ok=True)

    path = MODELS_DIR / "arima_arrivals_1_1_1.pkl"
    with open(path, "wb") as f:
        pickle.dump(baseline_arima_model, f)

    print("Saved ARIMA model →", path)


In [2]:
def evaluate_arima_order(series: pd.Series, order, val_months: int = 12):
    """
    Evaluate a single ARIMA(order) on last `val_months` points.
    Returns (rmse, mae).
    """
    if len(series) <= val_months + 5:
        print("[ARIMA] Not enough data for validation split.")
        return None, None

    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    try:
        model = ARIMA(train, order=order)
        fitted = model.fit()
        forecast_vals = fitted.forecast(steps=val_months)

        y_true = val.values
        y_pred = forecast_vals.values

        mse = np.mean((y_true - y_pred) ** 2)
        rmse = math.sqrt(mse)
        mae = np.mean(np.abs(y_true - y_pred))
        return rmse, mae
    except Exception as e:
        # Some (p,d,q) combos can fail; just skip them
        # print(f"Failed for order={order}: {e}")
        return None, None


In [3]:
series = df_arrivals["total_arrivals"]

p_values = [0, 1, 2, 3]
d_values = [0, 1, 2]
q_values = [0, 1, 2, 3]

best_order = None
best_rmse = float("inf")
best_mae = float("inf")

for p in p_values:
    for d in d_values:
        for q in q_values:
            order = (p, d, q)
            rmse, mae = evaluate_arima_order(series, order, val_months=12)
            if rmse is None:
                continue
            if rmse < best_rmse:
                best_rmse = rmse
                best_mae = mae
                best_order = order
                print(f"[ARIMA] New best order={order} RMSE={rmse:.2f} MAE={mae:.2f}")

print("\n=== Best ARIMA model on last 12 months ===")
print("order:", best_order)
print(f"RMSE : {best_rmse:.2f}")
print(f"MAE  : {best_mae:.2f}")


[ARIMA] New best order=(0, 0, 0) RMSE=199225.54 MAE=182421.14
[ARIMA] New best order=(0, 0, 1) RMSE=198352.88 MAE=181315.23
[ARIMA] New best order=(0, 0, 2) RMSE=186488.68 MAE=166395.50
[ARIMA] New best order=(0, 0, 3) RMSE=178025.53 MAE=161507.92
[ARIMA] New best order=(0, 1, 0) RMSE=140486.26 MAE=116421.50
[ARIMA] New best order=(0, 1, 1) RMSE=124936.37 MAE=102622.82


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


[ARIMA] New best order=(0, 1, 3) RMSE=120286.00 MAE=98490.52
[ARIMA] New best order=(0, 2, 2) RMSE=115222.75 MAE=91536.24


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


[ARIMA] New best order=(2, 1, 2) RMSE=99875.28 MAE=73752.77
[ARIMA] New best order=(2, 2, 3) RMSE=94900.02 MAE=71787.28


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


[ARIMA] New best order=(3, 2, 3) RMSE=94103.14 MAE=70504.72

=== Best ARIMA model on last 12 months ===
order: (3, 2, 3)
RMSE : 94103.14
MAE  : 70504.72


In [4]:
best_order = (3, 2, 3)

def evaluate_fixed_arima(series: pd.Series, order, val_months: int = 12):
    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    model = ARIMA(train, order=order)
    fitted = model.fit()
    forecast_vals = fitted.forecast(steps=val_months)

    y_true = val.values
    y_pred = forecast_vals.values

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = math.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    print(f"[ARIMA {order}] Validation window: {val.index.min().date()} → {val.index.max().date()}")
    print(f"[ARIMA {order}] Validation RMSE: {rmse:.2f}")
    print(f"[ARIMA {order}] Validation MAE : {mae:.2f}")

    return rmse, mae, fitted   # ✅ RETURN MODEL TOO


In [5]:
series = df_arrivals["total_arrivals"]
rmse_best, mae_best, arima_323_model = evaluate_fixed_arima(series, best_order, val_months=12)


[ARIMA (3, 2, 3)] Validation window: 2024-11-01 → 2025-10-01
[ARIMA (3, 2, 3)] Validation RMSE: 94103.14
[ARIMA (3, 2, 3)] Validation MAE : 70504.72


In [6]:
import pickle
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

save_path = MODELS_DIR / "arima_arrivals_3_2_3.pkl"

with open(save_path, "wb") as f:
    pickle.dump(arima_323_model, f)

print("Saved tuned ARIMA(3,2,3) →", save_path)


Saved tuned ARIMA(3,2,3) → models\arima_arrivals_3_2_3.pkl


In [15]:

best_arima_model = ARIMA(series, order=best_order).fit()

e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [16]:
def run_arima_forecast_with_order(df: pd.DataFrame, order, forecast_months: int = 12) -> pd.DataFrame:
    series = df["total_arrivals"]
    model = ARIMA(series, order=order)
    fitted = model.fit()
    forecast_values = fitted.forecast(steps=forecast_months)

    last_date = df.index.max()
    start_future = pd.to_datetime(last_date) + MonthBegin(1)
    future_index = pd.date_range(start=start_future, periods=forecast_months, freq="MS")

    future_df = pd.DataFrame({
        "date": future_index,
        "forecast_total_arrivals": forecast_values.values,
    })
    return future_df

future_df_best = run_arima_forecast_with_order(df_arrivals, best_order, forecast_months=12)
print(future_df_best.head())


        date  forecast_total_arrivals
0 2025-11-01            401882.769583
1 2025-12-01            417396.069178
2 2026-01-01            364416.109146
3 2026-02-01            362434.855015
4 2026-03-01            420650.818852


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
OUTPUT_CSV = FORECAST_DIR / "sltda_arrivals_arima_forecast.csv"

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
future_df_best.to_csv(OUTPUT_CSV, index=False)
print("Saved best ARIMA forecast →", OUTPUT_CSV)

Saved best ARIMA forecast → data_clean\forecast\sltda_arrivals_arima_forecast.csv


In [18]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

tuned_arima_path = MODELS_DIR / f"arima_arrivals_{best_order[0]}_{best_order[1]}_{best_order[2]}.pkl"

import pickle
with open(tuned_arima_path, "wb") as f:
    pickle.dump(best_arima_model, f)

print(f"Saved tuned ARIMA{best_order} →", tuned_arima_path)

Saved tuned ARIMA(3, 2, 3) → models\arima_arrivals_3_2_3.pkl


SARIMAAA


In [19]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def evaluate_sarima(series: pd.Series,
                    val_months: int = 12,
                    p_values=(0, 1, 2),
                    d_values=(0, 1),
                    q_values=(0, 1, 2),
                    P_values=(0, 1, 2),
                    D_values=(0, 1),
                    Q_values=(0, 1, 2),
                    seasonal_periods=12):
    """
    Simple grid search over a small SARIMA space.
    Returns (best_order, best_seasonal_order, best_rmse, best_mae, best_model).
    """
    if len(series) <= val_months + 5:
        print("[SARIMA] Not enough data for validation split.")
        return None, None, None, None, None

    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    best_rmse = float("inf")
    best_mae = float("inf")
    best_order = None
    best_seasonal = None
    best_model = None

    for p in p_values:
        for d in d_values:
            for q in q_values:
                for P in P_values:
                    for D in D_values:
                        for Q in Q_values:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, seasonal_periods)
                            try:
                                model = SARIMAX(
                                    train,
                                    order=order,
                                    seasonal_order=seasonal_order,
                                    enforce_stationarity=False,
                                    enforce_invertibility=False,
                                )
                                fitted = model.fit(disp=False)
                                forecast_vals = fitted.forecast(steps=val_months)

                                y_true = val.values
                                y_pred = forecast_vals.values
                                mse = np.mean((y_true - y_pred) ** 2)
                                rmse = math.sqrt(mse)
                                mae = np.mean(np.abs(y_true - y_pred))

                                if rmse < best_rmse:
                                    best_rmse = rmse
                                    best_mae = mae
                                    best_order = order
                                    best_seasonal = seasonal_order
                                    best_model = fitted
                                    print(
                                        f"[SARIMA] New best {order} x {seasonal_order} "
                                        f"RMSE={rmse:.2f} MAE={mae:.2f}"
                                    )
                            except Exception as e:
                                # Some combos are unstable; just skip
                                # print(f"Failed for {order} x {seasonal_order}: {e}")
                                continue

    print("\n=== Best SARIMA model ===")
    print("order         :", best_order)
    print("seasonal order:", best_seasonal)
    print(f"RMSE          : {best_rmse:.2f}")
    print(f"MAE           : {best_mae:.2f}")

    return best_order, best_seasonal, best_rmse, best_mae, best_model

series = df_arrivals["total_arrivals"]
best_order, best_seasonal, best_rmse, best_mae, best_model = evaluate_sarima(series, val_months=12)


[SARIMA] New best (0, 0, 0) x (0, 0, 0, 12) RMSE=395433.65 MAE=387239.50
[SARIMA] New best (0, 0, 0) x (0, 0, 1, 12) RMSE=233524.14 MAE=230672.06


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


[SARIMA] New best (0, 0, 0) x (0, 1, 0, 12) RMSE=59444.36 MAE=56812.33


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[SARIMA] New best (0, 0, 0) x (1, 1, 0, 12) RMSE=57374.69 MAE=54733.57


...

e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters ex

[SARIMA] New best (2, 1, 0) x (2, 1, 1, 12) RMSE=16737.75 MAE=15001.57


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations


=== Best SARIMA model ===
order         : (2, 1, 0)
seasonal order: (2, 1, 1, 12)
RMSE          : 16737.75
MAE           : 15001.57


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [20]:
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

def evaluate_sarima_refined(series: pd.Series,
                            val_months: int = 12):
    """
    Refined SARIMA search around the previously found good region.
    Returns (best_order, best_seasonal_order, best_rmse, best_mae, best_model).
    """
    if len(series) <= val_months + 5:
        print("[SARIMA] Not enough data for validation split.")
        return None, None, None, None, None

    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    # 🔍 Focused search space around (2,1,0)x(2,1,1,12)
    p_values = [1, 2, 3]
    d_values = [1]          # you already differenced once
    q_values = [0, 1, 2]

    P_values = [1, 2]
    D_values = [1]
    Q_values = [0, 1, 2]
    seasonal_periods = 12

    best_rmse = float("inf")
    best_mae = float("inf")
    best_order = None
    best_seasonal = None
    best_model = None

    for p in p_values:
        for d in d_values:
            for q in q_values:
                for P in P_values:
                    for D in D_values:
                        for Q in Q_values:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, seasonal_periods)
                            try:
                                with warnings.catch_warnings():
                                    warnings.filterwarnings("ignore")
                                    model = SARIMAX(
                                        train,
                                        order=order,
                                        seasonal_order=seasonal_order,
                                        enforce_stationarity=False,
                                        enforce_invertibility=False,
                                    )
                                    fitted = model.fit(disp=False, maxiter=200)

                                forecast_vals = fitted.forecast(steps=val_months)

                                y_true = val.values
                                y_pred = forecast_vals.values
                                mse = np.mean((y_true - y_pred) ** 2)
                                rmse = math.sqrt(mse)
                                mae = np.mean(np.abs(y_true - y_pred))

                                if rmse < best_rmse:
                                    best_rmse = rmse
                                    best_mae = mae
                                    best_order = order
                                    best_seasonal = seasonal_order
                                    best_model = fitted
                                    print(
                                        f"[SARIMA refined] New best {order} x {seasonal_order} "
                                        f"RMSE={rmse:.2f} MAE={mae:.2f}"
                                    )
                            except Exception:
                                continue

    print("\n=== Best SARIMA (refined) model ===")
    print("order         :", best_order)
    print("seasonal order:", best_seasonal)
    print(f"RMSE          : {best_rmse:.2f}")
    print(f"MAE           : {best_mae:.2f}")

    return best_order, best_seasonal, best_rmse, best_mae, best_model


In [21]:
series = df_arrivals["total_arrivals"]
best_order_ref, best_seasonal_ref, best_rmse_ref, best_mae_ref, best_model_ref = evaluate_sarima_refined(series, val_months=12)


[SARIMA refined] New best (1, 1, 0) x (1, 1, 0, 12) RMSE=25351.46 MAE=21864.57
[SARIMA refined] New best (1, 1, 0) x (1, 1, 1, 12) RMSE=24396.68 MAE=19117.56
[SARIMA refined] New best (1, 1, 1) x (2, 1, 2, 12) RMSE=24087.11 MAE=20338.54
[SARIMA refined] New best (2, 1, 0) x (2, 1, 0, 12) RMSE=18522.97 MAE=16106.90
[SARIMA refined] New best (2, 1, 0) x (2, 1, 1, 12) RMSE=16737.75 MAE=15001.57

=== Best SARIMA (refined) model ===
order         : (2, 1, 0)
seasonal order: (2, 1, 1, 12)
RMSE          : 16737.75
MAE           : 15001.57


In [ ]:
import os
import pickle

MODELS_DIR = REPO_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

sarima_path = MODELS_DIR / "sarima_arrivals_2_1_0_2_1_1_12.pkl"

with open(sarima_path, "wb") as f:
    pickle.dump(best_model_ref, f)

print("Saved SARIMA model to:", sarima_path)


Saved SARIMA model to: models\sarima_arrivals_2_1_0_2_1_1_12.pkl
